# Notebook19 — detached compute and gate pilot coordinator

Purpose: run the fixed two-seed, three-condition DEV comparison with bounded whole JEPA profiling. Input models use256pxRGB, physical16 accumulation8, the common four-worker ordered pipeline and rank64 learned features. The sole mathematical ablation sets initial local gate0.1 instead0.5. Outputs: artifacts/notebook_19/<RUN_ID>/report.html and learning/development_result.json. No official predictive pixels are opened.

Runtime: normally2–3hours after the discarded actual GPU resume proof. The immutable whole-study limit is10800seconds;900active seconds per arm, with a pinned200-update endpoint and initial controls. A busyGPU1 queues before the scientific clock starts. Other jobs are preserved. Performance is qualified by continuous1HzGPU observations separately from certified learning.

Checkpoint location: checkpoints/<RUN_ID>/stages/. The runner retains latest two valid resumable states, best and the pinned200-update checkpoint. Exact resume: open notebooks/19_Compute_Optimization_and_Global_Prior_Pilot.ipynb in a freshahr-gokernel, use the sameRUN_ID and clickRunAll at launch-compute-gate-pilot. Source, configuration and original clock must match; unmeasuredIN_FLIGHTwall time remainsUNKNOWN. Detached sessions fmm19-<RUN_ID>, -worker0 and -guard are independent of the user connection. Attach with tmux attach -t fmm19-<RUN_ID>; detach with Ctrl-b then d.

In [ ]:
"""Notebook19 detached DEV pilot; source-bound original clocks and one writer."""
from pathlib import Path
import copy,json,os,shlex,signal,subprocess,time,traceback
import nbformat
_SELF=Path(globals().get('__file__','/home/benjamin/workspace/ahr_go/fmm_jeppa/experiments/19_compute_and_gate/workflow.ipynb'))
_space={'__file__':str(_SELF.with_name('common.ipynb'))}
for _cell in nbformat.read(_SELF.with_name('common.ipynb'),4).cells:
    if _cell.cell_type=='code'and'ahr-export'in _cell.metadata.get('tags',[]):exec(compile(_cell.source,str(_SELF.with_name('common.ipynb')),'exec'),_space)
C=type('Common',(),_space)
MAIN='notebooks/19_Compute_Optimization_and_Global_Prior_Pilot.ipynb'
TERMINAL={'COMPLETE','UNKNOWN','FAILED','VALIDATION_COMPLETE'}
OPTIONS={'rank':64,'feature_seed':180064,'local_radius':1,'spatial_mode':'disabled','feature_temperature_min':.25,'feature_temperature_max':4.,'feature_temperature_initial':1.,'gate_initial':.5}
MAXIMUM_SECONDS=10800.;EVALUATION_RESERVE=3600.;TARGET_DEVICE='cuda:1'

def artifact_path(root,run_id):
    if not isinstance(run_id,str)or not run_id or any(not(c.isalnum()or c in'-_')for c in run_id):raise ValueError('Safe stable RUN_ID required')
    return Path(root).resolve()/'artifacts/notebook_19'/run_id

def session_names(run_id):
    prefix='fmm19-'+run_id
    return {key:prefix+suffix for key,suffix in [('coordinator',''),('guard','-guard'),('worker0','-worker0'),('launcher','-launcher')]}

def consumer_hash(root):
    return C.digest([c.source for c in nbformat.read(Path(root)/MAIN,4).cells if c.cell_type=='code'])

def status(artifact,state,phase,**details):
    value={'status':state,'phase':phase,'updated_unix':time.time(),**details}
    C.atomic_json(Path(artifact)/'study_status.json',value);print(state,phase,flush=True);return value

def report(root,artifact,result,profiling=None,evaluation=None,training=None):
    return C.load(root,'experiments/19_compute_and_gate/report.ipynb','fmm19_report').render(root,artifact,result,profiling,evaluation,training)

def available_devices(root):
    B=C.load(root,'experiments/17_fixed_kernel/benchmark.ipynb','fmm19_gpu_availability')
    try:
        observation=B._exclusive_gpu(TARGET_DEVICE)
        if not observation.get('eligible',False):raise RuntimeError('GPU observation did not certify conditional availability')
        return [TARGET_DEVICE],{TARGET_DEVICE:observation}
    except (RuntimeError,subprocess.SubprocessError,OSError)as exc:
        return [],{TARGET_DEVICE:{'status':'UNAVAILABLE','reason':repr(exc),'existing_jobs_preserved':True}}

def validate_queue_registration(root,artifact):
    value=C.read(Path(artifact)/'queue_registration.json')
    if value['sources']!=C.source_hashes(root)or value['consumer_hash']!=consumer_hash(root):raise ValueError('Queued source or frontdoor changed; refuse silent redesign')
    if value['options']!=OPTIONS or value['target_device']!=TARGET_DEVICE or value['seeds']!=[20262201,20262202]:raise ValueError('Queued candidate, target or fixed seeds changed')
    return value

def queue_registration(root,artifact,run_id):
    path=Path(artifact)/'queue_registration.json'
    if path.exists():return validate_queue_registration(root,artifact)
    value={'schema':'notebook19-queued-launch/v1','run_id':run_id,'requested_unix':time.time(),'sources':C.source_hashes(root),
      'consumer_hash':consumer_hash(root),'options':OPTIONS,'target_device':TARGET_DEVICE,'seeds':[20262201,20262202],
      'queue_policy':'Wait before discarded GPUproof and scientific3hclock. No foreign job is stopped. Queuedsource and design remain immutable.',
      'official_test':'CLOSED','maximum_scientific_seconds':MAXIMUM_SECONDS}
    return C.immutable(path,value)

def prepare_launch(root,run_id):
    root=Path(root);R=C.runner(root);sources=C.source_hashes(root)
    proof_artifact=artifact_path(root,run_id+'-launch-preflight-'+C.digest(sources)[:10]);proof_artifact.mkdir(parents=True,exist_ok=True)
    path=root/'artifacts/notebook_19_validation/operator/validation_receipt.json'
    operator=C.read(path)
    if operator.get('status')!='PASS'or operator.get('actual_cuda')!='PASS'or operator.get('check_count',0)<106 or operator.get('operator_export_sha256')!=C.export_hash(root/'experiments/19_compute_and_gate/operator.ipynb'):raise ValueError('Current106-check CPU/CUDA output/allgradient gate required')
    result_path=proof_artifact/'actual_preflight_result.json'
    if result_path.exists():
        proof=C.read(result_path)
        if proof.get('source_hashes')!=sources:raise ValueError('Cached actualproof sources changed')
        # A completed original proof can be reused after its short deadline.
        # Revalidate all saved reference/split manifests and final hashes first.
        backend,_=R.bind(root,OPTIONS)
        for arm in proof.get('arms',{}).values():
            if set(arm.get('checkpoints',{}))!={'reference','resumed'}:raise ValueError('Cachedproof reference and resumed checkpoints both required')
            for record in arm['checkpoints'].values():
                directory=Path(record['directory']).resolve()
                if not directory.is_relative_to(root/'checkpoints'/proof_artifact.name):raise ValueError('Proof checkpoint escaped its run')
                manifest=validate_manifest_rotation(backend,directory)
                if manifest.get('successful_updates')!=4 or manifest.get('attempted_updates')!=4:raise ValueError('Cachedproof must retain complete exact4-update states')
                if C.file_record(directory/manifest['latest'])['sha256']!=record['sha256']:raise ValueError('Cached resumableproof checkpoint hash changed')
    else:proof=R.actual_preflight(root,proof_artifact,OPTIONS,256,4,device=TARGET_DEVICE)
    arms=proof.get('arms',{})
    if proof.get('image_size')!=256 or proof.get('input_workers')!=4:raise ValueError('Actualproof resolution or ordered pipeline changed')
    plan_record=proof.get('plan',{})
    if not plan_record.get('path')or not plan_record.get('sha256'):raise ValueError('Actualproof plan record path and hash required')
    actual_plan_record=C.file_record(plan_record['path'])
    if Path(actual_plan_record['path']).resolve()!=Path(plan_record['path']).resolve()or actual_plan_record['sha256']!=plan_record['sha256']or('bytes'in plan_record and actual_plan_record['bytes']!=plan_record['bytes']):raise ValueError('Actualproof plan record hash or declared size changed')
    plan_path=Path(plan_record['path']).resolve()
    if plan_path!=proof_artifact/'actual_preflight_plan.json':raise ValueError('Actualproof plan path escaped its run')
    proof_plan=C.read(plan_path)
    if proof_plan.get('run_id')!=proof_artifact.name or proof_plan.get('source_hashes')!=sources or proof_plan.get('devices')!=[TARGET_DEVICE] or proof_plan.get('input_workers')!=4 or proof_plan.get('selected')!={**OPTIONS,'image_size':256}:raise ValueError('Actualproof plan identity changed')
    if proof_plan.get('updates')!=4000 or any(a['config'].get('image_size')!=256 or a['config']['training'].get('microbatch')!=16 or a['config']['training'].get('accumulation_steps')!=8 for a in proof_plan.get('arms',[])):raise ValueError('Actualproof model batch or scheduler configuration changed')
    if proof.get('status')!='PASS'or not proof.get('passed')or proof.get('source_hashes')!=sources or len(arms)!=3 or not all(v.get('status')=='PASS'and v.get('bitwise_equal')for v in arms.values()):raise ValueError('Actual3-condition256px16/8 complete exact-resume proof required')
    required=('vit','learned_optimized','learned_global_prior')
    if not all(any(key.endswith('-'+variant)for key in arms)for variant in required):raise ValueError('Actualproof dropped a condition')
    gate_path=proof_artifact/'operator_numerical_gate.json'
    gate={'status':'PASS','passed':True,'source_hashes':sources,'operator_source_sha256':operator['operator_export_sha256'],
      'receipt':C.file_record(path),'check_count':operator['check_count'],'actual_cuda':True,
      'evidence_role':'Independent own-kernel output/allgradient and completeJEPA equivalence only; no quality or speed gate'}
    C.immutable(gate_path,gate);return C.file_record(gate_path),C.file_record(result_path)

def check_registration(root,artifact):
    req=C.read(Path(artifact)/'launch_registration.json');clock=C.read(Path(artifact)/'clock.json')
    if req['sources']!=C.source_hashes(root)or req['consumer_hash']!=consumer_hash(root):raise ValueError('Frozen19 source/frontdoor changed')
    if clock!=req['clock']or C.file_record(Path(artifact)/'clock.json')!=req['clock_record']:raise ValueError('Original3hour clock changed')
    if clock['maximum_seconds']!=MAXIMUM_SECONDS or abs(clock['deadline_unix']-clock['started_unix']-MAXIMUM_SECONDS)>1e-5:raise ValueError('Original deadline must never reset or extend')
    if req['devices']!=[TARGET_DEVICE]or req['options']!=OPTIONS or req['seeds']!=[20262201,20262202]:raise ValueError('Registered19 devices/options/seeds changed')
    if req['active_wall_seconds_per_arm']!=900 or req['matched_updates']!=200 or req['evaluation_reserve_seconds']!=EVALUATION_RESERVE or req['input_workers']!=4 or req['gpu_workers']!=1:raise ValueError('Registered wall, milestone, reserve or pipeline changed')
    for key in('operator_gate','preflight'):
        if C.file_record(req[key]['path'])!=req[key]:raise ValueError('Launch numerical or resume evidence changed')
    return req,clock

def run_validation(root,run_id='compute-gate-frontdoor-validation'):
    artifact=artifact_path(root,run_id);artifact.mkdir(parents=True,exist_ok=True)
    result={'status':'VALIDATION_COMPLETE','fixture_source':'SOFTWARE_PRESENTATION_ONLY; NO_SCIENTIFIC_TRAINING','official_test':'CLOSED','run_id':run_id}
    report(root,artifact,result);status(artifact,'VALIDATION_COMPLETE','software_presentation');return result

def launch_study(root,run_id='compute-gate-001',profile=None,allow_queue=True):
    root=Path(root).resolve();C.bootstrap(root);profile=profile or os.environ.get('FMM19_PROFILE','full')
    if profile=='validation':return run_validation(root,run_id)
    if profile!='full':raise ValueError('Full or validation profile required')
    artifact=artifact_path(root,run_id);artifact.mkdir(parents=True,exist_ok=True)
    from fmm_jepa.food101_training import run_writer
    with run_writer(artifact/'launch_writer'):
        existing=C.read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
        if existing.get('status')in TERMINAL and not (artifact/'launch_registration.json').exists():
            if (artifact/'queue_registration.json').exists():validate_queue_registration(root,artifact)
            return {**existing,'report':str(artifact/'report.html'),'resume':'Terminal receipt retained; no silent restart'}
        if (artifact/'launch_registration.json').exists():
            req,clock=check_registration(root,artifact)
            if existing.get('status')in TERMINAL:return {**existing,'report':str(artifact/'report.html'),'resume':'Terminalreceipt retained; no silent restart'}
        else:
            if (artifact/'clock.json').exists():raise ValueError('Orphan originalclock; inspection required, never restart')
            queue_registration(root,artifact,run_id);devices,observations=available_devices(root)
            if not devices:
                status(artifact,'QUEUED','waiting_for_conditional_GPU_availability',device_observations=observations)
                if allow_queue:
                    sessions=session_names(run_id);entry=write_operator_notebook(root,artifact/'operators/launcher.ipynb','run_queue_launcher',{'root':str(root),'run_id':run_id})
                    process=spawn(root,entry,sessions['launcher'],artifact/'launcher.log')
                else:process={'status':'ALREADY_IN_QUEUE_LAUNCHER'}
                receipt={'status':'QUEUED','run_id':run_id,'device':TARGET_DEVICE,'queue':process,'scientific_clock_started':False,
                  'official_test':'CLOSED','artifact':str(artifact),'resume':'SameNotebook19RUN_ID and unchangedsources; queue has no scientific clock until actualproof passes'}
                C.atomic_json(artifact/'queue_receipt.json',receipt);return receipt
            operator,proof=prepare_launch(root,run_id)
            devices,observations=available_devices(root)
            if not devices:
                status(artifact,'QUEUED','GPU_became_active_after_discarded_proof',device_observations=observations)
                if allow_queue:
                    sessions=session_names(run_id);entry=write_operator_notebook(root,artifact/'operators/launcher.ipynb','run_queue_launcher',{'root':str(root),'run_id':run_id})
                    spawn(root,entry,sessions['launcher'],artifact/'launcher.log')
                return {'status':'QUEUED','run_id':run_id,'scientific_clock_started':False,'reason':'GPU active after proof; same queue launcher waits'}
            started=time.time();clock={'started_unix':started,'deadline_unix':started+MAXIMUM_SECONDS,'maximum_seconds':MAXIMUM_SECONDS,
              'scope':'Preregistration, boundedsyntheticwholeJEPAprofile,6arms,initial/matched/finalDEVprobes and checkpoints; noofficialreadout',
              'restart_policy':'No deadline reset or extension'}
            C.immutable(artifact/'clock.json',clock)
            req={'schema':'notebook19-launch/v1','run_id':run_id,'sources':C.source_hashes(root),'consumer_hash':consumer_hash(root),'clock':clock,'clock_record':C.file_record(artifact/'clock.json'),
              'operator_gate':operator,'preflight':proof,'options':OPTIONS,'devices':devices,'device_observations':observations,'input_workers':4,
              'seeds':[20262201,20262202],'active_wall_seconds_per_arm':900.,'matched_updates':200,'evaluation_reserve_seconds':EVALUATION_RESERVE,'gpu_workers':1,
              'device_policy':'OnlyCUDA1. ThreeidleSMsamplespermitresidentidlecontextsconditionally; continuous1Hzobservationqualifiesactualperformance separately fromlearning.',
              'official_test':'CLOSED','training_gate':'Ownkernel numericalvalidity and actualfullstate resume only; no mimicry,speed or DEVqualityselection',
              'additional_online_parameters':98328,'parameter_matching':'Sharedoriginalweights/predictor paired; learnedcandidates have98328extraonlineparameters',
              'inference':'Twofreshpairedtriplets developmentpilot; no significance or superiority claim'}
            C.immutable(artifact/'launch_registration.json',req)
        C.check_deadline(clock['deadline_unix']);sessions=session_names(run_id)
        if not (artifact/'report.html').exists():report(root,artifact,{'status':'RUNNING','phase':'launch','clock':clock,'official_test':'CLOSED','run_id':run_id})
        guard=write_operator_notebook(root,artifact/'operators/guard.ipynb','run_guard',{'root':str(root),'run_id':run_id})
        coordinator=write_operator_notebook(root,artifact/'operators/coordinator.ipynb','run_coordinator',{'root':str(root),'run_id':run_id})
        a=spawn(root,guard,sessions['guard'],artifact/'guard.log');b=spawn(root,coordinator,sessions['coordinator'],artifact/'coordinator.log')
        receipt={'status':'DETACHED_PIPELINE_STARTED','run_id':run_id,'clock':clock,'sessions':sessions,'guard':a,'coordinator':b,
          'artifact':str(artifact),'report':str(artifact/'report.html'),'attach':'tmux attach -t '+sessions['coordinator'],'detach':'Ctrl-b then d',
          'resume':'Freshahr-gokernel, sameRUN_ID, Notebook19RunAll, unchangedsources/options/originalclock. IN_FLIGHT timers stayUNKNOWN; never forgive unmeasuredtime.'}
        C.atomic_json(artifact/'launch_receipt.json',receipt);return receipt

def run_queue_launcher(root,run_id):
    root=Path(root);artifact=artifact_path(root,run_id)
    try:
        while True:
            validate_queue_registration(root,artifact)
            result=launch_study(root,run_id,'full',allow_queue=False)
            if result['status']!='QUEUED':return result
            C.atomic_json(artifact/'queue_heartbeat.json',{'updated_unix':time.time(),'status':'QUEUED','scientific_clock_started':False,'target_device':TARGET_DEVICE})
            time.sleep(10)
    except BaseException as exc:
        status(artifact,'UNKNOWN','queued_source_or_launch_failure',reason=repr(exc));return {'status':'UNKNOWN','reason':repr(exc),'official_test':'CLOSED'}

def gate_receipts(req):
    return [{'role':role,'status':'PASS','passed':True,'evidence':req[key],'evidence_role':'NUMERICAL_OR_RESUME_ONLY; NO_ACCURACY_OR_SPEED_SELECTION'}for role,key in [('operator','operator_gate'),('exact_resume','preflight')]]

def worker_folder(artifact,worker):
    if worker!='worker0':raise ValueError('Notebook19onlyregisteredworker0')
    return Path(artifact)/'learning/workers'/worker

def run_worker(root,run_id,worker):
    root=Path(root);artifact=artifact_path(root,run_id);req,clock=check_registration(root,artifact);folder=worker_folder(artifact,worker);R=C.runner(root)
    plan=C.read(folder/'plan.json');P=C.load(root,'experiments/19_compute_and_gate/profiling.ipynb','fmm19_worker_observation');observation=P.ContinuousGPUObservation(TARGET_DEVICE)
    try:
        status(folder,'RUNNING','data_and_exact_resume');C.environment(plan['arms'][0]['config']);bundle=R.data_bundle(root,folder/'data',plan['arms'][0]['config'],verify_images=False)
        observation.start()
        try:training=R.train_worker(root,folder,plan,bundle,clock['deadline_unix']-EVALUATION_RESERVE)
        finally:
            observation.stop();foreign=[s for s in observation.samples if s['pid']not in(None,os.getpid())and(s['sm_percent']or 0)>0]
            qualification={'status':'UNKNOWN_FOREIGN_COMPUTE'if foreign else'OBSERVED_NO_FOREIGN_COMPUTE','own_pid':os.getpid(),'target_device':TARGET_DEVICE,
              'samples':observation.samples,'foreign_active_samples':foreign,'scope':'Continuous1Hzobservation coverspreflight andtraining. Subsecondsamplinggapsremainpossible; certifiedlearning ispreservedifforeigncompute occurs.'}
            C.atomic_json(folder/'gpu_performance_qualification.json',qualification)
        training['performance_qualification']=C.file_record(folder/'gpu_performance_qualification.json');C.atomic_json(folder/'training_result.json',training)
        if training.get('status')!='COMPLETE':status(folder,'UNKNOWN','training_fairness');return training
        status(folder,'RUNNING','initial_matched_final_DEV_readouts')
        evaluation=R.evaluate_worker(root,folder,plan,bundle,training,clock['deadline_unix']-60)
        state='COMPLETE'if evaluation.get('status')=='COMPLETE'else'UNKNOWN';status(folder,state,'finished')
        return {'status':state,'training':str(folder/'training_result.json'),'evaluation':str(folder/'evaluation_result.json'),'official_test':'CLOSED'}
    except BaseException as exc:
        status(folder,'UNKNOWN','exception',reason=repr(exc),traceback=traceback.format_exc());raise

def merge_worker_results(plan,training_parts,evaluation_parts):
    if len(training_parts)!=1 or len(evaluation_parts)!=1:raise ValueError('Exactly one registered worker result required')
    expected={a['id']for a in plan['arms']};training={'status':'COMPLETE','fair_wall_budget_verified':True,'arms':{},'performance_qualifications':[]};groups={'rows':[],'initial_rows':[],'matched_rows':[]}
    for t,e in zip(training_parts,evaluation_parts):
        if t.get('status')!='COMPLETE'or not t.get('fair_wall_budget_verified')or e.get('status')!='COMPLETE':raise ValueError('Alltriplettraining andprimaryreadoutsrequired')
        if set(t['arms'])&set(training['arms']):raise ValueError('Duplicateworkerarm')
        training['arms'].update(t['arms']);training['performance_qualifications'].append(t.get('performance_qualification'))
        for key in groups:groups[key].extend(e.get(key,[]))
    if set(training['arms'])!=expected or len(expected)!=6:raise ValueError('All6registeredarmsrequired')
    for key,rows in groups.items():
        identifiers=[row['arm_id']for row in rows]
        if len(identifiers)!=len(set(identifiers))or set(identifiers)!=expected:raise ValueError('Endpointconditiondroppedorduplicated:'+key)
        for row in rows:
            arm=next(a for a in plan['arms']if a['id']==row['arm_id'])
            if int(row['seed'])!=int(arm['seed'])or row['variant']!=arm['variant']:raise ValueError('Endpointseed/conditionmembershipchanged')
    values=list(training['arms'].values());tol=max(plan['wall_tolerance_seconds'],plan['active_wall_seconds_per_arm']*plan['wall_tolerance_fraction'])
    if max(v['active_wall_seconds']for v in values)-min(v['active_wall_seconds']for v in values)>tol:raise ValueError('Equalactivewall6-armfairnessrequired')
    evaluation={'status':'COMPLETE',**groups,'matched_status':'COMPLETE'if all(r.get('primary_readout_status')=='COMPLETE'for r in groups['matched_rows'])else'UNKNOWN','official_test':'CLOSED','source_hashes':plan['source_hashes']}
    return training,evaluation

def launch_wait_workers(root,artifact,req,plan):
    R=C.runner(root);sessions=session_names(req['run_id']);folder=worker_folder(artifact,'worker0');folder.mkdir(parents=True,exist_ok=True)
    child=R.worker_plan(plan,'worker0');child['stage_id']='pilot-worker0';C.immutable(folder/'plan.json',child)
    state=C.read(folder/'study_status.json')if(folder/'study_status.json').exists()else{}
    if state.get('status')in('UNKNOWN','FAILED'):raise ValueError('Priorincompleteworker; preserveUNKNOWN')
    if state.get('status')!='COMPLETE':
        notebook=write_operator_notebook(root,folder/'worker.ipynb','run_worker',{'root':str(root),'run_id':req['run_id'],'worker':'worker0'})
        spawn(root,notebook,sessions['worker0'],folder/'worker.log')
    started=time.time()
    while True:
        C.check_deadline(req['clock']['deadline_unix']);state=C.read(folder/'study_status.json')if(folder/'study_status.json').exists()else{'status':'STARTING'}
        if state['status']in('UNKNOWN','FAILED'):raise RuntimeError('Workerincomplete:'+str(state))
        if state['status']=='COMPLETE':break
        if time.time()-started>90 and not session_exists(sessions['worker0']):raise RuntimeError('Ownedworkersessiondisappeared; preserveoriginaltiming')
        C.atomic_json(Path(artifact)/'coordinator_heartbeat.json',{'updated_unix':time.time(),'phase':'pilot_training_and_DEV_readouts','worker_state':state,'deadline_unix':req['clock']['deadline_unix']});time.sleep(10)
    training,evaluation=merge_worker_results(plan,[C.read(folder/'training_result.json')],[C.read(folder/'evaluation_result.json')])
    C.atomic_json(Path(artifact)/'learning/training_result.json',training);C.atomic_json(Path(artifact)/'learning/evaluation_result.json',evaluation)
    terminate_session_tree(sessions['worker0']);return training,evaluation

def run_coordinator(root,run_id):
    root=Path(root);artifact=artifact_path(root,run_id);req,clock=check_registration(root,artifact);profiling=None;evaluation=None;training=None
    from fmm_jepa.food101_training import run_writer
    try:
        with run_writer(artifact/'coordinator_writer'):
            R=C.runner(root);F=C.load(root,'experiments/19_compute_and_gate/protocol.ipynb','fmm19_protocol');folder=artifact/'learning';folder.mkdir(parents=True,exist_ok=True)
            path=folder/'plan.json';plan=C.read(path)if path.exists()else R.make_screen_plan(root,run_id,req['options'],req['devices'],clock['deadline_unix'],4,gate_receipts(req))
            C.immutable(path,plan);bundle=R.data_bundle(root,folder/'data',plan['arms'][0]['config'],verify_images=True)
            F.register(root,folder,plan,bundle)
            status(artifact,'RUNNING','bounded_whole_JEPA_profile')
            P=C.load(root,'experiments/19_compute_and_gate/profiling.ipynb','fmm19_scientific_profile');activation=artifact/'profiling_activation.json'
            if activation.exists():profile_clock=C.read(activation)
            else:
                started=time.time();profile_clock={'started_unix':started,'deadline_unix':min(clock['deadline_unix']-EVALUATION_RESERVE,started+600),'maximum_seconds':600,'source_hashes':req['sources']};C.immutable(activation,profile_clock)
            if profile_clock['source_hashes']!=req['sources']or profile_clock['deadline_unix']>clock['deadline_unix']-EVALUATION_RESERVE:raise ValueError('Originalprofileactivationchanged')
            cached=artifact/'profiling/profile_result.json'
            if cached.exists():profiling=C.read(cached)
            else:
                try:profiling=P.run_profile(root,artifact/'profiling',device=TARGET_DEVICE,deadline=profile_clock['deadline_unix'])
                except (TimeoutError,RuntimeError,ValueError)as exc:
                    profiling={'status':'UNKNOWN','reason':repr(exc),'official_test_pixels_read':0,'source_hashes':req['sources'],'deadline_unix':profile_clock['deadline_unix']};C.atomic_json(cached,profiling)
            # Computeperformance andtracequalityareexplanatory, neverlearninggates.
            report(root,artifact,{'status':'RUNNING','phase':'two_fresh_triplets_full_JEPA','official_test':'CLOSED','run_id':run_id},profiling)
            status(artifact,'RUNNING','two_fresh_triplets_full_JEPA')
            training,evaluation=launch_wait_workers(root,artifact,req,plan)
            result=F.analyze(plan,evaluation,training,profiling);result['clock']=clock
            qualifications=[C.read(v['path'])for v in training.get('performance_qualifications',[])if v]
            result['training_performance_status']='UNKNOWN'if any(q.get('status')!='OBSERVED_NO_FOREIGN_COMPUTE'for q in qualifications)or len(qualifications)!=1 else'OBSERVED_NO_FOREIGN_COMPUTE'
            C.atomic_json(folder/'development_result.json',result);report(root,artifact,result,profiling,evaluation,training)
            status(artifact,result['status'],'finished',signals=result.get('signals'),report=str(artifact/'report.html'));return result
    except BaseException as exc:
        stopped=terminate_session_tree(session_names(run_id)['worker0']);result={'status':'UNKNOWN','reason':repr(exc),'traceback':traceback.format_exc(),'official_test':'CLOSED','stopped_owned_worker':stopped,'run_id':run_id}
        report(root,artifact,result,profiling,evaluation,training);status(artifact,'UNKNOWN','cutoff_or_integrity_failure',reason=repr(exc));return result

def run_guard(root,run_id):
    root=Path(root);artifact=artifact_path(root,run_id);sessions=session_names(run_id);last_review=0.;missing_since=None
    try:
        while True:
            req,clock=check_registration(root,artifact);existing=C.read(artifact/'study_status.json')if(artifact/'study_status.json').exists()else{}
            if existing.get('status')in TERMINAL:return {'status':'GUARD_COMPLETE','study_status':existing['status']}
            now=time.time()
            if not session_exists(sessions['coordinator']):
                if missing_since is None:missing_since=now
                elif now-missing_since>90:raise RuntimeError('Coordinatorabsentbeyondstartupgrace')
            else:missing_since=None
            if now>=clock['deadline_unix']:raise TimeoutError('Original3hourdeadlineexhausted')
            C.atomic_json(artifact/'guard_heartbeat.json',{'status':'HEALTHY','updated_unix':now,'deadline_unix':clock['deadline_unix'],'sessions':{k:session_exists(v)for k,v in sessions.items()}})
            if now-last_review>=3600:
                R,_,_=C.modules(root);audits=[]
                for manifest in(root/'checkpoints'/run_id).glob('stages/*/arms/*/manifest.json'):
                    current=validate_manifest_rotation(R,manifest.parent);audits.append({'path':str(manifest.parent),'successful_updates':current['successful_updates'],'latest_valid':True})
                C.atomic_json(artifact/'hourly_review.json',{'status':'PASS','updated_unix':now,'audits':audits,'source_and_original_clock_valid':True});last_review=now
            time.sleep(10)
    except BaseException as exc:
        stopped={key:terminate_session_tree(sessions[key])for key in('coordinator','worker0')};result={'status':'UNKNOWN','reason':repr(exc),'stopped_owned_project_processes':stopped,'official_test':'CLOSED','run_id':run_id}
        report(root,artifact,result);status(artifact,'UNKNOWN','guard_deadline_or_integrity_failure',reason=repr(exc));return result


def session_exists(name):return subprocess.run(['tmux','has-session','-t','='+name],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL).returncode==0

def terminate_session_tree(session):
    """Stop only processes descending from this exact project-owned tmux session."""
    if not session_exists(session):return []
    roots={int(x)for x in subprocess.check_output(['tmux','list-panes','-t','='+session,'-F','#{pane_pid}'],text=True).split()}
    edges=[tuple(map(int,line.split()))for line in subprocess.check_output(['ps','-eo','pid=,ppid='],text=True).splitlines()];owned=set(roots)
    while True:
        expanded=owned|{pid for pid,parent in edges if parent in owned}
        if expanded==owned:break
        owned=expanded
    for pid in sorted(owned-roots,reverse=True)+sorted(roots):
        try:os.kill(pid,signal.SIGKILL)
        except ProcessLookupError:pass
    subprocess.run(['tmux','kill-session','-t','='+session],capture_output=True);return sorted(owned)

def validate_manifest_rotation(runner,directory):
    for attempt in range(4):
        try:return runner.validate_training_manifest(directory)
        except (FileNotFoundError,ValueError):
            if attempt==3:raise
            time.sleep(.25)

def write_operator_notebook(root,path,function,arguments):
    code="from pathlib import Path\nimport json,nbformat,types,os\nos.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')\nROOT=Path("+repr(str(Path(root).resolve()))+")\nP=ROOT/'experiments/19_compute_and_gate/workflow.ipynb'\nW=types.ModuleType('fmm19_detached_workflow');W.__file__=str(P)\nfor cell in nbformat.read(P,4).cells:\n    if cell.cell_type=='code'and'ahr-export'in cell.metadata.get('tags',[]):exec(compile(cell.source,str(P),'exec'),W.__dict__)\nRESULT=W."+function+"(**json.loads("+repr(json.dumps(arguments))+"))\nprint(json.dumps(RESULT,indent=2,default=str))\n"
    nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Detached19 '+function+'\nGenerated operator entrypoint. Frozen parent source, original deadline and single-writer checkpoints remain authoritative. Resume through Notebook19 with the same RUN_ID.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'name':'ahr-go','display_name':'AHR FMM-JEPA','language':'python'}})
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        old=nbformat.read(path,4)
        if [c.source for c in old.cells if c.cell_type=='code']!=[c.source for c in nb.cells if c.cell_type=='code']:raise ValueError('Detached entrypoint code changed')
    else:nbformat.write(nb,path)
    return path

def spawn(root,path,session,log):
    if session_exists(session):return {'status':'EXISTING_SESSION','session':session}
    jupyter=Path(root).parent/'.venv/bin/jupyter';command=' '.join(shlex.quote(str(x))for x in[jupyter,'nbconvert','--to','notebook','--execute','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',path])+' >> '+shlex.quote(str(log))+' 2>&1'
    env='CUBLAS_WORKSPACE_CONFIG=:4096:8 OMP_NUM_THREADS=2 MKL_NUM_THREADS=2 '
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),env+command],check=True)
    return {'status':'STARTED','session':session,'operator_notebook':str(path),'log':str(log)}


In [ ]:
print('Notebook19 workflow definitions ready. The main notebook launches or queues the authorized bounded pilot.')